In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# 1. 인증 정보 설정
CLIENT_ID = "CLIENT_IDD" 
CLIENT_SECRET = "CLIENT_SECRETT"

headers = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET,
    "Content-Type": "application/json"
}

#날짜 설정
str_start_date = "2021-01-01" 
str_end_date = "2026-01-31"

# --- [기능 1: 검색어 트렌드] ---
def get_search_trend(keywords_group):
    # API 주소가 변경되었습니다 (openapi.naver.com 사용 권장)
    url = "https://openapi.naver.com/v1/datalab/search"
    body = {
        "startDate": str_start_date, # 자동 설정된 날짜 적용
        "endDate": str_end_date,
        "timeUnit": "week",
        "keywordGroups": keywords_group,
        "ages": ["3", "4", "5", "6"] 
    }
    response = requests.post(url, headers=headers, json=body)
    return response.json()

# --- [기능 2: 쇼핑인사이트] ---
def get_shopping_insight(category_id, keyword_list):
    url = "https://openapi.naver.com/v1/datalab/shopping/category/keywords"
    body = {
    "startDate": str_start_date,
    "endDate": str_end_date,
    "timeUnit": "week",
    "category": category_id,
    "keywords": keyword_list,
    "ages": ["3", "4", "5", "6"],
    # 아래 항목들을 추가하면 더 자세한 통계가 담깁니다
    "gender": "",   # 빈값은 전체, "f"는 여성, "m"은 남성
    "device": ""    # 빈값은 전체, "pc"는 PC, "mo"는 모바일
}
    response = requests.post(url, headers=headers, json=body)
    return response.json()

# --- [기능 3: 통합 검색] ---
def get_search_result(query, display=10):
    # query 변수를 사용하도록 수정
    url = f"https://openapi.naver.com/v1/search/blog?query={query}&display={display}"
    response = requests.get(url, headers=headers)
    return response.json()

# --- [실행 파트] ---

# 1. 지역별 트렌드 그룹 설정
trend_groups = [
    {"groupName": "성수", "keywords": ["성수동 팝업", "성수 팝업","성수 핫플"]},
    {"groupName": "강남", "keywords": ["강남역 팝업", "강남 팝업","강남 핫플"]},
    {"groupName": "한남", "keywords": ["한남동 팝업", "한남 팝업","한남 핫플"]},
    {"groupName": "홍대", "keywords": ["홍대 팝업", "연남동 팝업","홍대 핫플"]}
]

# 실행 및 결과 확인
trend_data = get_search_trend(trend_groups)
print("1. 검색어 트렌드 결과:", trend_data.get('results', '에러 발생'))

# 2. 쇼핑 인사이트 추가 테스트 (예: 패션의류 카테고리)
shopping_keywords = [{"패션의류": "원피스", "param": ["50000000"]}]
shopping_data = get_shopping_insight("50000000", shopping_keywords)
print("\n3. 쇼핑 인사이트 결과:", shopping_data.get('results', '에러 발생'))

blog_data = get_search_result("팝업스토어")
print("\n2. 블로그 검색 결과 (첫 번째 글 제목):", blog_data['items'][0]['title'] if 'items' in blog_data else "결과 없음")

In [ ]:
# --- 트렌드 데이터 변환 및 저장 ---
trend_list = []
for group in trend_data.get('results', []):
    group_name = group['title']
    for entry in group['data']:
        trend_list.append({
            "date": entry['period'],
            "region": group_name,
            "ratio": entry['ratio']
        })

df_trend = pd.DataFrame(trend_list)
# 엑셀에서 보기 편하게 피벗(가로로 펼치기)
df_trend_pivot = df_trend.pivot(index='date', columns='region', values='ratio')
df_trend_pivot.to_csv("naver_trend_results.csv", encoding="utf-8-sig")
print("검색어 트렌드 CSV 저장 완료!")
# --- 쇼핑 데이터 변환 및 저장 ---
shop_list = []
for group in shopping_data.get('results', []):
    keyword_name = group['title']
    for entry in group['data']:
        shop_list.append({
            "date": entry['period'],
            "keyword": keyword_name,
            "click_ratio": entry['ratio']
        })

df_shop = pd.DataFrame(shop_list)
df_shop.to_csv("naver_shopping_results.csv", index=False, encoding="utf-8-sig")
print("쇼핑 인사이트 CSV 저장 완료!")
# --- 블로그 데이터 변환 및 저장 ---
blog_list = []
for item in blog_data.get('items', []):
    blog_list.append({
        "title": item['title'].replace('<b>', '').replace('</b>', ''), # 태그 제거
        "link": item['link'],
        "description": item['description'].replace('<b>', '').replace('</b>', ''),
        "postdate": item['postdate']
    })

df_blog = pd.DataFrame(blog_list)
df_blog.to_csv("naver_blog_results.csv", index=False, encoding="utf-8-sig")
print("블로그 검색 결과 CSV 저장 완료!")

In [ ]:
# 검색어트렌드
import requests
import pandas as pd
from datetime import datetime, timedelta

# 1. 인증 정보 설정
CLIENT_ID = "CLIENT_IDD" 
CLIENT_SECRET = "CLIENT_SECRETT"

headers = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET,
    "Content-Type": "application/json"
}

#날짜 설정
str_start_date = "2021-01-01" 
str_end_date = "2026-01-31"

# --- [기능 1: 검색어 트렌드] ---
def get_search_trend(keywords_group):
    # API 주소가 변경되었습니다 (openapi.naver.com 사용 권장)
    url = "https://openapi.naver.com/v1/datalab/search"
    body = {
        "startDate": str_start_date, # 자동 설정된 날짜 적용
        "endDate": str_end_date,
        "timeUnit": "date",
        "keywordGroups": keywords_group,
    }
    response = requests.post(url, headers=headers, json=body)
    return response.json()
trend_groups = [
    {"groupName": "성수", "keywords": ["성수동 팝업", "성수 팝업","성수 핫플"]},
    {"groupName": "강남", "keywords": ["강남역 팝업", "강남 팝업","강남 핫플"]},
    {"groupName": "한남", "keywords": ["한남동 팝업", "한남 팝업","한남 핫플"]},
    {"groupName": "홍대", "keywords": ["홍대 팝업", "연남동 팝업","홍대 핫플"]}
]
trend_data = get_search_trend(trend_groups)
print("1. 검색어 트렌드 결과:", trend_data.get('results', '에러 발생'))
trend_list = []
for group in trend_data.get('results', []):
    group_name = group['title']
    for entry in group['data']:
        trend_list.append({
            "date": entry['period'],
            "region": group_name,
            "ratio": entry['ratio']
        })
df_trend = pd.DataFrame(trend_list)
# 엑셀에서 보기 편하게 피벗(가로로 펼치기)
df_trend_pivot = df_trend.pivot(index='date', columns='region', values='ratio')
# df_trend_pivot.to_csv("naver_trend_results.csv", encoding="utf-8-sig")
# print("검색어 트렌드 CSV 저장 완료!")